Bloc 1 — Imports et configuration

In [1]:
import os
import pandas as pd
import numpy as np
import sqlite3
import logging
import warnings
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger(__name__)

BASE        = Path(r"C:\Users\admin\Downloads\MSPR2 (4) (6)\MSPR2")
MASTER_CSV  = BASE / "03_processed" / "master_ml.csv"
DB_PATH     = BASE / "03_database"  / "mspr2.db"
MODELS_DIR  = BASE / "04_outputs"   / "models"
OUTPUTS_DIR = BASE / "04_outputs"   / "ml"

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

REGROUPEMENT = {
    "Extreme gauche"      : "Gauche",
    "Gauche radicale"     : "Gauche",
    "Gauche"              : "Gauche",
    "Ecologie"            : "Gauche",
    "Centre"              : "Centre",
    "Droite"              : "Droite",
    "Droite souverainiste": "Droite",
    "Extreme droite"      : "Extreme droite",
    "Divers"              : "Divers",
    "Inconnu"             : "Divers",
}

# Colonnes socioeconomiques stables par commune
COLS_SOCIO = [
    "population", "densite", "superficie_km2",
    "nb_logements", "nb_residences_principales", "nb_logements_vacants",
    "actifs_2022", "chomeurs_2022", "actifs_occupes_2022", "inactifs_2022",
    "taux_chomage_2022", "revenu_median", "taux_pauvrete",
    "part_chomeurs", "part_foyers_imposes",
    "d1_revenu", "d9_revenu", "ratio_interdecile",
    "nb_immigres", "nb_non_immigres", "total_delits",
    "nb_menages", "nb_personnes_menages"
]

# Scores europeennes
COLS_EURO = [
    "euro_centre_ratio", "euro_divers_ratio", "euro_droite_ratio",
    "euro_extreme_droite_ratio", "euro_extreme_gauche_ratio",
    "euro_gauche_ratio", "euro_gauche_radicale_ratio",
    "euro_droite_souverainiste_ratio", "euro_ecologie_ratio",
    "euro_taux_participation"
]

PermissionError: [WinError 5] Accès refusé: 'C:\\Users\\admin'

Bloc 2 — Chargement et regroupement

In [ ]:
if MASTER_CSV.exists():
    df = pd.read_csv(MASTER_CSV, sep=";", low_memory=False)
else:
    conn = sqlite3.connect(DB_PATH)
    df   = pd.read_sql("SELECT * FROM master_ml", conn)
    conn.close()

df["famille_regroupee"] = df["famille_gagnante"].map(REGROUPEMENT).fillna("Divers")

print("Distribution par election et famille regroupee :")
print(df.groupby(["id_election", "famille_regroupee"]).size().unstack(fill_value=0).to_string())

In [ ]:
print("═" * 60)
for name, res in results.items():
    print(f"{name}")
    print(f"  CV Accuracy  : {res['cv_accuracy']}")
    print(f"  Test Accuracy: {res['test_accuracy']}")
    print(f"  F1 (weighted): {res['f1_weighted']}")
    print(f"  Params       : {res['best_params']}")
    print()
print(f" Meilleur modèle : {best_name} (acc={best_acc:.4f})")

Bloc 3 — Construction du nouveau master par commune

In [ ]:
def extraire_scores_election(df, id_election, suffixe):
    df_elec = df[df["id_election"] == id_election][
        ["code_commune", "famille_regroupee"] + COLS_SOCIO + COLS_EURO
    ].copy()

    # Renommer les colonnes avec le suffixe de l'election
    rename = {
        "famille_regroupee": f"famille_{suffixe}"
    }
    for col in COLS_EURO:
        rename[col] = f"{col}_{suffixe}"
    for col in COLS_SOCIO:
        rename[col] = f"{col}_{suffixe}"

    df_elec = df_elec.rename(columns=rename)
    return df_elec


# Extraction par election
df_2017_legi = extraire_scores_election(df, "2017_legi_t1", "legi_2017")
df_2017_pres = extraire_scores_election(df, "2017_pres_t1", "pres_2017")
df_2022_legi = extraire_scores_election(df, "2022_legi_t1", "legi_2022")
df_2022_pres = extraire_scores_election(df, "2022_pres_t1", "pres_2022")
df_2024_legi = extraire_scores_election(df, "2024_legi_t1", "legi_2024")

# Variable cible : famille gagnante aux legislatives 2024
df_cible = df_2024_legi[["code_commune", "famille_legi_2024"]].copy()
df_cible = df_cible.rename(columns={"famille_legi_2024": "cible_2024"})

# Jointure de toutes les elections sur code_commune
df_nouveau_master = df_cible.copy()

for df_elec, label in [
    (df_2017_legi, "legi_2017"),
    (df_2017_pres, "pres_2017"),
    (df_2022_legi, "legi_2022"),
    (df_2022_pres, "pres_2022"),
]:
    cols_a_joindre = ["code_commune"] + [
        c for c in df_elec.columns
        if c != "code_commune"
    ]
    df_nouveau_master = df_nouveau_master.merge(
        df_elec[cols_a_joindre],
        on="code_commune",
        how="left"
    )

# Calcul des deltas entre 2017 et 2022 pour capturer l'evolution
for col in COLS_EURO:
    col_2017 = f"{col}_legi_2017"
    col_2022 = f"{col}_legi_2022"
    if col_2017 in df_nouveau_master.columns and col_2022 in df_nouveau_master.columns:
        df_nouveau_master[f"delta_{col}"] = (
            df_nouveau_master[col_2022] - df_nouveau_master[col_2017]
        )

print(f"Nouveau master : {len(df_nouveau_master):,} lignes | {df_nouveau_master.shape[1]} colonnes")
print(f"\nVariable cible - distribution :")
print(df_nouveau_master["cible_2024"].value_counts().to_string())
print(f"\nNaN par colonne :")
nan_counts = df_nouveau_master.isna().sum()
print(nan_counts[nan_counts > 0].to_string() if nan_counts.any() else "  aucune")

Bloc 4 — Preparation des features et split

In [ ]:
# Construction de X et y
cols_exclure_master = ["code_commune", "cible_2024",
                       "famille_legi_2017", "famille_pres_2017",
                       "famille_legi_2022", "famille_pres_2022"]

cols_features = [c for c in df_nouveau_master.columns if c not in cols_exclure_master]

X = df_nouveau_master[cols_features].copy()
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    if X[col].isna().sum() > 0:
        X[col] = X[col].fillna(X[col].median())

le_cible = LabelEncoder()
y        = le_cible.fit_transform(df_nouveau_master["cible_2024"].fillna("Divers"))

print(f"Features : {X.shape[1]}")
print(f"Communes : {X.shape[0]}")
print(f"Classes  : {list(le_cible.classes_)}")
print(f"\nDistribution de la cible :")
for cls, count in zip(*np.unique(y, return_counts=True)):
    print(f"  {le_cible.classes_[cls]:<20} {count} ({count/len(y)*100:.1f}%)")

# Split stratifie 80/20
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y
)

print(f"\nTrain : {len(X_train):,} communes")
print(f"Test  : {len(X_test):,}  communes")

print(f"\nDistribution train :")
for cls, count in zip(*np.unique(y_train, return_counts=True)):
    print(f"  {le_cible.classes_[cls]:<20} {count}")

print(f"\nDistribution test :")
for cls, count in zip(*np.unique(y_test, return_counts=True)):
    print(f"  {le_cible.classes_[cls]:<20} {count}")

Bloc 5 — Definition des modeles

In [ ]:
# On utilise StratifiedKFold au lieu de TimeSeriesSplit
# car on n'a plus de dimension temporelle - une seule ligne par commune
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

modeles = {
    "Random Forest": {
        "pipeline": Pipeline([
            ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
        ]),
        "params": {
            "clf__n_estimators"     : [100, 200, 300, 500],
            "clf__max_depth"        : [None, 5, 10, 20, 30],
            "clf__min_samples_split": [2, 5, 10],
            "clf__min_samples_leaf" : [1, 2, 4],
            "clf__max_features"     : ["sqrt", "log2", None],
            "clf__class_weight"     : [None, "balanced"],
        }
    },
    "Gradient Boosting": {
        "pipeline": Pipeline([
            ("clf", GradientBoostingClassifier(random_state=42))
        ]),
        "params": {
            "clf__n_estimators"     : [100, 200, 300],
            "clf__learning_rate"    : [0.01, 0.05, 0.1, 0.2],
            "clf__max_depth"        : [3, 5, 7, 10],
            "clf__subsample"        : [0.6, 0.8, 1.0],
            "clf__min_samples_split": [2, 5, 10],
            "clf__max_features"     : ["sqrt", "log2", None],
        }
    },
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                random_state=42, max_iter=1000, n_jobs=-1
            ))
        ]),
        "params": {
            "clf__C"           : [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__penalty"     : ["l1", "l2", "elasticnet"],
            "clf__solver"      : ["saga"],
            "clf__l1_ratio"    : [0.0, 0.25, 0.5, 0.75, 1.0],
            "clf__class_weight": [None, "balanced"],
        }
    },
    "KNN": {
        "pipeline": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_jobs=-1))
        ]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
            "clf__weights"    : ["uniform", "distance"],
            "clf__metric"     : ["euclidean", "manhattan", "minkowski"],
            "clf__p"          : [1, 2],
        }
    },
}

print(f"{len(modeles)} modeles definis.")

Bloc 6 — Entrainement

In [ ]:
resultats = {}

for nom, config in modeles.items():
    log.info(f"Entrainement : {nom}...")

    search = RandomizedSearchCV(
        estimator           = config["pipeline"],
        param_distributions = config["params"],
        n_iter              = 30,
        cv                  = skf,
        scoring             = "f1_weighted",
        n_jobs              = -1,
        random_state        = 42,
        refit               = True,
    )

    search.fit(X_train, y_train)

    resultats[nom] = {
        "modele"  : search.best_estimator_,
        "params"  : search.best_params_,
        "score_cv": round(search.best_score_, 4),
    }

    log.info(f"  Score CV : {search.best_score_:.4f}")
    log.info(f"  Params   : {search.best_params_}")

Bloc 7 — Evaluation

In [ ]:
bilan = []

for nom, res in resultats.items():
    y_pred = res["modele"].predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    f1     = f1_score(y_test, y_pred, average="weighted", zero_division=0)

    bilan.append({
        "modele"     : nom,
        "accuracy"   : round(acc, 4),
        "f1_weighted": round(f1, 4),
        "score_cv"   : res["score_cv"],
    })

    print(f"\n{nom}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  F1 weighted : {f1:.4f}")
    print(f"  Score CV    : {res['score_cv']:.4f}")
    print(classification_report(
        y_test, y_pred,
        target_names=le_cible.classes_,
        zero_division=0
    ))

df_bilan = pd.DataFrame(bilan).sort_values("f1_weighted", ascending=False)
display(df_bilan)

Bloc 8 — Matrices de confusion

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (nom, res) in enumerate(resultats.items()):
    y_pred = res["modele"].predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(
        cm_pct,
        annot=True, fmt=".1f",
        xticklabels=le_cible.classes_,
        yticklabels=le_cible.classes_,
        cmap="Blues", ax=axes[i],
        cbar_kws={"label": "%"}
    )
    axes[i].set_title(
        f"{nom}\nAccuracy={accuracy_score(y_test, y_pred):.3f} | "
        f"F1={f1_score(y_test, y_pred, average='weighted', zero_division=0):.3f}",
        fontsize=11, fontweight="bold"
    )
    axes[i].set_xlabel("Prediction")
    axes[i].set_ylabel("Reel")
    axes[i].tick_params(axis="x", rotation=20)

plt.suptitle("Matrices de confusion - nouveau master par commune",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / "ml_nouveau_master_confusion.png", bbox_inches="tight")
plt.show()